In [ ]:
#pip install requests pandas

In [ ]:
#pip install schedule

In [ ]:
#pip install pyodbc

In [ ]:
import pandas as pd
import pyodbc

SERVER_NAME = r'DESKTOP-2A0QBS7\SQLEXPRESS'
DATABASE_NAME = 'EgyptTrafficDB'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;'


df_points = pd.read_csv('egypt_traffic_points.csv')

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

count = 0
for index, row in df_points.iterrows():
  
    cursor.execute("SELECT LocationID FROM Locations WHERE LocationName = ?", (row['LocationName'],))
    if not cursor.fetchone():
        cursor.execute(
            "INSERT INTO Locations (LocationName, City, Latitude, Longitude) VALUES (?, ?, ?, ?)",
            (row['LocationName'], row['City'], float(row['Latitude']), float(row['Longitude']))
        )
        count += 1

conn.commit()
conn.close()
print(f"تم بنجاح إدخال {count} نقطة جديدة إلى قاعدة البيانات! 📍")

In [ ]:
import time
from datetime import datetime
import requests
import pyodbc
import schedule

SERVER_NAME = r'DESKTOP-2A0QBS7\SQLEXPRESS'
DATABASE_NAME = 'EgyptTrafficDB'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;'


API_KEY = "5cUh0bqAswvmUlzjXzwg2hAS7nDoG87L"

def run_traffic_collection():
    print(f"\n--- بدء دورة سحب المرور الجديدة في: {datetime.now()} ---")
    
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    

    cursor.execute("SELECT LocationID, Latitude, Longitude, LocationName FROM Locations")
    locations = cursor.fetchall()
    
    success_count = 0
    for loc in locations:
        loc_id, lat, lon, name = loc
        url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat}%2C{lon}&key={API_KEY}"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                flow_data = data['flowSegmentData']
                current_speed = flow_data['currentSpeed']
                normal_speed = flow_data['freeFlowSpeed']
                
                # تحديد حالة الزحمة
                if current_speed < (normal_speed * 0.6):
                    status = "Heavy Congestion"
                elif current_speed < normal_speed:
                    status = "Moderate"
                else:
                    status = "Free Flow"
                
                # إدخال السجل في جدول Traffic_Logs
                insert_query = """
                    INSERT INTO Traffic_Logs (LocationID, CurrentSpeed, NormalSpeed, TrafficStatus, RecordedAt)
                    VALUES (?, ?, ?, ?, ?)
                """
                cursor.execute(insert_query, (loc_id, current_speed, normal_speed, status, datetime.now()))
                conn.commit()
                success_count += 1
            else:
                print(f"فشل سحب نقطة {name}: {response.status_code}")
        except Exception as e:
            print(f"خطأ في نقطة {name}: {e}")
      
        time.sleep(0.5)
        
    conn.close()
    print(f"خلصت دورة السحب بنجاح. تم تسجيل {success_count} نقطة في الـ DB.")


# 1. 05:00 ص
schedule.every().day.at("05:00").do(run_traffic_collection)
# 2. 08:00 ص
schedule.every().day.at("08:00").do(run_traffic_collection)
# 3. 12:00 م
schedule.every().day.at("12:00").do(run_traffic_collection)
# 4. 05:00 م
schedule.every().day.at("17:00").do(run_traffic_collection)
# 5. 10:00 م
schedule.every().day.at("22:00").do(run_traffic_collection)
# 6. 02:00 ص
schedule.every().day.at("02:00").do(run_traffic_collection)

print("الـ Traffic Pipeline شغالة في الخلفية وجاهزة في المواعيد المحددة!")


while True:
    schedule.run_pending()
    time.sleep(60)

In [2]:
import time
from datetime import datetime
import requests
import pyodbc

SERVER_NAME = r'DESKTOP-2A0QBS7\SQLEXPRESS'
DATABASE_NAME = 'EgyptTrafficDB'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;'


API_KEY = "5cUh0bqAswvmUlzjXzwg2hAS7nDoG87L"

def run_traffic_collection_once():
    print(f"\n--- بدء دورة السحب الفوري في: {datetime.now()} ---")
    
    try:
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        
      
        cursor.execute("SELECT LocationID, Latitude, Longitude, LocationName FROM Locations")
        locations = cursor.fetchall()
        
        if not locations:
            print("لا توجد نقاط مسجلة في جدول Locations!")
            return
            
        success_count = 0
        for loc in locations:
            loc_id, lat, lon, name = loc
            url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat}%2C{lon}&key={API_KEY}"
            
            try:
                response = requests.get(url)
                if response.status_code == 200:
                    data = response.json()
                    flow_data = data['flowSegmentData']
                    current_speed = flow_data['currentSpeed']
                    normal_speed = flow_data['freeFlowSpeed']
                    
          
                    if current_speed < (normal_speed * 0.6):
                        status = "Heavy Congestion"
                    elif current_speed < normal_speed:
                        status = "Moderate"
                    else:
                        status = "Free Flow"
                    
             
                    insert_query = """
                        INSERT INTO Traffic_Logs (LocationID, CurrentSpeed, NormalSpeed, TrafficStatus, RecordedAt)
                        VALUES (?, ?, ?, ?, ?)
                    """
                    cursor.execute(insert_query, (loc_id, current_speed, normal_speed, status, datetime.now()))
                    conn.commit()
                    success_count += 1
                else:
                    print(f"فشل سحب نقطة {name}: {response.status_code}")
            except Exception as e:
                print(f"خطأ في نقطة {name}: {e}")
                
       
            time.sleep(0.5)
            
        conn.close()
        print(f"تم الانتهاء بنجاح! تم تسجيل {success_count} نقطة في الـ DB.")
        
    except Exception as e:
        print(f"خطأ في الاتصال بقاعدة البيانات: {e}")


run_traffic_collection_once()


--- بدء دورة السحب الفوري في: 2026-08-31 18:14:04.932515 ---
تم الانتهاء بنجاح! تم تسجيل 101 نقطة في الـ DB.
